In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if cwd.name == 'url_corpus' and cwd.parent.name == 'notebooks':
    os.chdir(cwd.parent)

nb_dir = Path.cwd().resolve()
backend_root = nb_dir.parent if nb_dir.name == 'notebooks' else nb_dir
if str(backend_root) not in sys.path:
    sys.path.insert(0, str(backend_root))

print(f'Notebook CWD: {Path.cwd()}')
print(f'Backend root in sys.path: {backend_root}')


# Kompleksowe testowanie wytrenowanych modeli

Ten notebook zawiera testowanie wszystkich wytrenowanych modeli na dedykowanych zbiorach testowych.


## 1. Import bibliotek i konfiguracja


In [ ]:
import sys
cwd = Path.cwd().resolve(); os.chdir(cwd.parent) if cwd.name == 'url_corpus' and cwd.parent.name == 'notebooks' else None; nb_dir = Path.cwd().resolve(); backend_root = nb_dir.parent if nb_dir.name == 'notebooks' else nb_dir; sys.path.insert(0, str(backend_root)) if str(backend_root) not in sys.path else None

import warnings
warnings.filterwarnings('ignore')

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
from utils.feature_names import get_readable_name

MODELS_DIR = Path('../models')
TEST_DATA_DIR = Path('../data/url_corpus/test')
RESULTS_DIR = Path('../data/url_corpus/phase_testing_results')
PLOTS_DIR = Path('../static/plots_url_corpus/phase_testing')

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

for i in range(1, 6):
    (RESULTS_DIR / f'phase_{i}').mkdir(parents=True, exist_ok=True)
    (PLOTS_DIR / f'phase_{i}').mkdir(parents=True, exist_ok=True)

(PLOTS_DIR / 'summary').mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'summary').mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.titlesize'] = 16

np.random.seed(42)

print('✅ Biblioteki zaimportowane pomyślnie!')
print(f'\n📁 Struktura katalogów:')
print(f'  • Wyniki: {RESULTS_DIR}/')
print(f'  • Wykresy: {PLOTS_DIR}/')
print(f'  • Utworzono podfoldery dla 5 faz oraz folder summary')

## 2. Wczytanie zbiorów testowych


In [ ]:
test_full = pd.read_csv(TEST_DATA_DIR / 'test_dataset_full.csv')
test_optimized = pd.read_csv(TEST_DATA_DIR / 'test_dataset_optimized.csv')

test_datasets = {
    'full': test_full,
    'optimized': test_optimized
}

print('='*80)
print('WCZYTANE ZBIORY TESTOWE')
print('='*80)
for name, df in test_datasets.items():
    print(f'\n{name.upper()}:')
    print(f'  Rozmiar: {df.shape}')
    print(f'  Liczba cech: {df.shape[1]-1}')
    print(f'  Phishing: {(df.phishing==1).sum()} ({(df.phishing==1).sum()/len(df)*100:.1f}%)')
    print(f'  Bezpieczne: {(df.phishing==0).sum()} ({(df.phishing==0).sum()/len(df)*100:.1f}%)')

## 3. Definicja struktury modeli i ładowanie


In [ ]:
model_configs = [
    {
        'name': 'Regresja Logistyczna',
        'dir': 'logistic_regression',
        'model_pattern': 'logistic_regression_{}_url_corpus.pkl',
        'scaler_pattern': 'logistic_regression_scaler_{}_url_corpus.pkl',
        'has_scaler': True,
        'is_dict': False
    },
    {
        'name': 'Random Forest',
        'dir': 'random_forest',
        'model_pattern': 'random_forest_{}_url_corpus.pkl',
        'scaler_pattern': None,
        'has_scaler': False,
        'is_dict': True
    },
    {
        'name': 'XGBoost',
        'dir': 'xgboost',
        'model_pattern': 'xgboost_model_{}_url_corpus.pkl',
        'scaler_pattern': None,
        'has_scaler': False,
        'is_dict': True
    },
    {
        'name': 'SVM',
        'dir': 'svm',
        'model_pattern': 'svm_model_{}_url_corpus.pkl',
        'scaler_pattern': None,
        'has_scaler': False,
        'is_dict': True
    },
    {
        'name': 'MLP',
        'dir': 'mlp',
        'model_pattern': 'mlp_model_{}_url_corpus.pkl',
        'scaler_pattern': None,
        'has_scaler': False,
        'is_dict': True
    }
]

loaded_models = {}

print('='*80)
print('ŁADOWANIE MODELI')
print('='*80)

for config in model_configs:
    model_name = config['name']
    model_dir = MODELS_DIR / config['dir']

    loaded_models[model_name] = {}

    for dataset_type in ['full', 'optimized']:
        model_path = model_dir / config['model_pattern'].format(dataset_type)

        if model_path.exists():
            with open(model_path, 'rb') as f:
                loaded_data = pickle.load(f)

            if config['is_dict']:
                model = loaded_data['model']
                scaler = loaded_data.get('scaler', None)
            else:
                model = loaded_data
                scaler = None

                if config['has_scaler']:
                    scaler_path = model_dir / config['scaler_pattern'].format(dataset_type)
                    if scaler_path.exists():
                        with open(scaler_path, 'rb') as f:
                            scaler = pickle.load(f)

            loaded_models[model_name][dataset_type] = {
                'model': model,
                'scaler': scaler
            }
            print(f'✅ {model_name} ({dataset_type}): załadowano')
        else:
            print(f'❌ {model_name} ({dataset_type}): brak pliku {model_path.name}')

print(f'\n✅ Załadowano {sum(len(v) for v in loaded_models.values())} modeli')


## 4. Funkcje pomocnicze


In [ ]:
def prepare_test_data(df):
    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]
    return X, y

def predict_with_model(model, scaler, X):
    if scaler is not None:
        X_scaled = scaler.transform(X)
    else:
        X_scaled = X.values

    predictions = model.predict(X_scaled)

    if hasattr(model, 'predict_proba'):
        probabilities = model.predict_proba(X_scaled)[:, 1]
    elif hasattr(model, 'decision_function'):
        probabilities = model.decision_function(X_scaled)
        probabilities = 1 / (1 + np.exp(-probabilities))
    else:
        probabilities = predictions.flatten()

    if len(predictions.shape) > 1:
        predictions = (predictions > 0.5).astype(int).flatten()

    return predictions, probabilities

def calculate_metrics(y_true, y_pred, y_proba):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_proba),
        'confusion_matrix': confusion_matrix(y_true, y_pred)
    }

def analyze_misclassifications(X, y_true, y_pred, y_proba, indices):
    misclassified = []

    for idx in indices:
        if y_true.iloc[idx] != y_pred[idx]:
            misclassified.append({
                'index': idx,
                'true_label': y_true.iloc[idx],
                'predicted_label': y_pred[idx],
                'probability': y_proba[idx],
                'features': X.iloc[idx].to_dict()
            })

    return misclassified

print('✅ Funkcje pomocnicze zdefiniowane')


## 5. Testowanie modeli w 5 fazach z różnymi rozkładami danych


In [ ]:
def create_phase_dataset(df, n_samples, phishing_ratio):
    phishing_count = int(n_samples * phishing_ratio)
    legitimate_count = n_samples - phishing_count

    phishing_available = (df['phishing'] == 1).sum()
    legitimate_available = (df['phishing'] == 0).sum()

    if phishing_count > phishing_available:
        phishing_samples = df[df['phishing'] == 1].sample(n=phishing_available, random_state=42, replace=False)
        phishing_samples = pd.concat([phishing_samples] * (phishing_count // phishing_available + 1), ignore_index=True)
        phishing_samples = phishing_samples.sample(n=phishing_count, random_state=42, replace=False)
    else:
        phishing_samples = df[df['phishing'] == 1].sample(n=phishing_count, random_state=42, replace=False)

    if legitimate_count > legitimate_available:
        legitimate_samples = df[df['phishing'] == 0].sample(n=legitimate_available, random_state=42, replace=False)
        legitimate_samples = pd.concat([legitimate_samples] * (legitimate_count // legitimate_available + 1), ignore_index=True)
        legitimate_samples = legitimate_samples.sample(n=legitimate_count, random_state=42, replace=False)
    else:
        legitimate_samples = df[df['phishing'] == 0].sample(n=legitimate_count, random_state=42, replace=False)

    phase_df = pd.concat([phishing_samples, legitimate_samples], ignore_index=True)
    phase_df = phase_df.sample(frac=1, random_state=42).reset_index(drop=True)

    return phase_df

test_phases = [
    {
        'name': 'Faza 1: Mini Test',
        'size': 2,
        'distributions': [
            {'phishing': 0.5, 'label': '50% phishing, 50% bezpieczne'}
        ]
    },
    {
        'name': 'Faza 2: 1,000 próbek',
        'size': 1000,
        'distributions': [
            {'phishing': 0.7, 'label': '70% phishing, 30% bezpieczne'},
            {'phishing': 0.3, 'label': '30% phishing, 70% bezpieczne'},
            {'phishing': 0.5, 'label': '50% phishing, 50% bezpieczne'}
        ]
    },
    {
        'name': 'Faza 3: 5,000 próbek',
        'size': 5000,
        'distributions': [
            {'phishing': 0.7, 'label': '70% phishing, 30% bezpieczne'},
            {'phishing': 0.3, 'label': '30% phishing, 70% bezpieczne'},
            {'phishing': 0.5, 'label': '50% phishing, 50% bezpieczne'}
        ]
    },
    {
        'name': 'Faza 4: 10,000 próbek',
        'size': 10000,
        'distributions': [
            {'phishing': 0.7, 'label': '70% phishing, 30% bezpieczne'},
            {'phishing': 0.3, 'label': '30% phishing, 70% bezpieczne'},
            {'phishing': 0.5, 'label': '50% phishing, 50% bezpieczne'}
        ]
    },
    {
        'name': 'Faza 5: 20,000 próbek',
        'size': 20000,
        'distributions': [
            {'phishing': 0.7, 'label': '70% phishing, 30% bezpieczne'},
            {'phishing': 0.3, 'label': '30% phishing, 70% bezpieczne'},
            {'phishing': 0.5, 'label': '50% phishing, 50% bezpieczne'}
        ]
    }
]

phase_results = {}

print('='*100)
print('TESTOWANIE FAZOWE MODELI')
print('='*100)

for phase_idx, phase in enumerate(test_phases, 1):
    print(f'\n{"="*100}')
    print(f'{phase["name"].upper()}')
    print(f'{"="*100}')

    phase_results[phase['name']] = {}

    for model_name, datasets in loaded_models.items():
        phase_results[phase['name']][model_name] = {}

        for dataset_type, model_data in datasets.items():
            phase_results[phase['name']][model_name][dataset_type] = {}

            test_df = test_datasets[dataset_type]
            model = model_data['model']
            scaler = model_data['scaler']

            print(f'\n{model_name} - {dataset_type.upper()}')
            print('-'*100)

            for dist in phase['distributions']:
                dist_df = create_phase_dataset(test_df, phase['size'], dist['phishing'])
                X_test, y_test = prepare_test_data(dist_df)

                predictions, probabilities = predict_with_model(model, scaler, X_test)
                metrics = calculate_metrics(y_test, predictions, probabilities)
                misclassified = analyze_misclassifications(
                    X_test, y_test, predictions, probabilities, range(len(y_test))
                )

                phase_results[phase['name']][model_name][dataset_type][dist['label']] = {
                    'metrics': metrics,
                    'misclassified': misclassified,
                    'distribution': dist
                }

                print(f"  {dist['label']}:")
                print(f"    Dokładność: {metrics['accuracy']:.4f} | Precyzja: {metrics['precision']:.4f} | "
                      f"Czułość: {metrics['recall']:.4f} | F1: {metrics['f1']:.4f} | "
                      f"ROC-AUC: {metrics['roc_auc']:.4f} | Błędy: {len(misclassified)}/{len(y_test)}")

print(f'\n{"="*100}')
print('✅ Testowanie fazowe zakończone')
print(f'{"="*100}')

## 6. Podsumowanie wyników fazowych - szczegółowe tabele


In [ ]:
phase_summary_data = []

for phase_name, models in phase_results.items():
    for model_name, datasets in models.items():
        for dataset_type, distributions in datasets.items():
            for dist_label, results in distributions.items():
                metrics = results['metrics']
                phase_summary_data.append({
                    'Faza': phase_name,
                    'Model': model_name,
                    'Zbior': 'Pelny (111)' if dataset_type == 'full' else 'Zoptymalizowany (73)',
                    'Rozklad': dist_label,
                    'Dokladnosc': metrics['accuracy'],
                    'Precyzja': metrics['precision'],
                    'Czulosc': metrics['recall'],
                    'F1': metrics['f1'],
                    'ROC-AUC': metrics['roc_auc'],
                    'Bledne': len(results['misclassified'])
                })

phase_summary_df = pd.DataFrame(phase_summary_data)

print('='*120)
print('SZCZEGOLOWE PODSUMOWANIE WYNIKOW FAZOWYCH')
print('='*120)

phase_mapping = {
    'Faza 1: Mini Test': 1,
    'Faza 2: 1,000 probek': 2,
    'Faza 2: 1,000 próbek': 2,
    'Faza 3: 5,000 probek': 3,
    'Faza 3: 5,000 próbek': 3,
    'Faza 4: 10,000 probek': 4,
    'Faza 4: 10,000 próbek': 4,
    'Faza 5: 20,000 probek': 5,
    'Faza 5: 20,000 próbek': 5
}

for dataset_type in ['Pelny (111)', 'Zoptymalizowany (73)']:
    print(f'\n{"#"*120}')
    print(f'WYNIKI DLA ZBIORU: {dataset_type}')
    print(f'{"#"*120}')

    dataset_data = phase_summary_df[phase_summary_df['Zbior'] == dataset_type]

    for phase_name in dataset_data['Faza'].unique():
        phase_data = dataset_data[dataset_data['Faza'] == phase_name]
        phase_data_sorted = phase_data.sort_values(['ROC-AUC', 'F1'], ascending=False)

        print(f'\n{phase_name} - {dataset_type}')
        print('-'*100)
        display_cols = ['Model', 'Rozklad', 'Dokladnosc', 'Precyzja', 'Czulosc', 'F1', 'ROC-AUC', 'Bledne']
        print(phase_data_sorted[display_cols].to_string(index=False))

        phase_num = phase_mapping[phase_name]
        dataset_suffix = 'full' if 'Pelny' in dataset_type else 'optimized'
        csv_path = RESULTS_DIR / f'phase_{phase_num}' / f'phase_{phase_num}_{dataset_suffix}_results.csv'
        phase_data_sorted.to_csv(csv_path, index=False)
        print(f'Zapisano do: {csv_path}')

csv_all_path = RESULTS_DIR / 'summary' / 'all_phases_results.csv'
phase_summary_df.to_csv(csv_all_path, index=False)

print(f'\n{"="*120}')
print(f'Zapisano wszystkie wyniki do: {csv_all_path}')
print(f'Zapisano {len(phase_summary_df)} wynikow w {len(phase_mapping)*2} plikach fazowych')
print(f'{"="*120}')


print('\n' + '='*120)
print('PODSUMOWANIE WEDLUG MODELI - SREDNIE METRYKI')
print('='*120)

print('\n' + '#'*80)
print('PELNY ZBIOR DANYCH (111 cech)')
print('#'*80)
full_dataset_summary = phase_summary_df[phase_summary_df['Zbior'] == 'Pelny (111)'].groupby('Model').agg({
    'Dokladnosc': ['mean', 'std'],
    'Precyzja': ['mean', 'std'],
    'Czulosc': ['mean', 'std'],
    'F1': ['mean', 'std'],
    'ROC-AUC': ['mean', 'std'],
    'Bledne': 'sum'
}).round(4)

full_dataset_flat = pd.DataFrame()
for model in full_dataset_summary.index:
    full_dataset_flat.loc[model, 'Dokladnosc (sr.)'] = full_dataset_summary.loc[model, ('Dokladnosc', 'mean')]
    full_dataset_flat.loc[model, 'Dokladnosc (std)'] = full_dataset_summary.loc[model, ('Dokladnosc', 'std')]
    full_dataset_flat.loc[model, 'Precyzja (sr.)'] = full_dataset_summary.loc[model, ('Precyzja', 'mean')]
    full_dataset_flat.loc[model, 'Precyzja (std)'] = full_dataset_summary.loc[model, ('Precyzja', 'std')]
    full_dataset_flat.loc[model, 'Czulosc (sr.)'] = full_dataset_summary.loc[model, ('Czulosc', 'mean')]
    full_dataset_flat.loc[model, 'Czulosc (std)'] = full_dataset_summary.loc[model, ('Czulosc', 'std')]
    full_dataset_flat.loc[model, 'F1 (sr.)'] = full_dataset_summary.loc[model, ('F1', 'mean')]
    full_dataset_flat.loc[model, 'F1 (std)'] = full_dataset_summary.loc[model, ('F1', 'std')]
    full_dataset_flat.loc[model, 'ROC-AUC (sr.)'] = full_dataset_summary.loc[model, ('ROC-AUC', 'mean')]
    full_dataset_flat.loc[model, 'ROC-AUC (std)'] = full_dataset_summary.loc[model, ('ROC-AUC', 'std')]
    full_dataset_flat.loc[model, 'Suma bledow'] = full_dataset_summary.loc[model, ('Bledne', 'sum')]

full_dataset_flat = full_dataset_flat.sort_values('ROC-AUC (sr.)', ascending=False)
print(full_dataset_flat.round(4))

print('\n' + '#'*80)
print('ZOPTYMALIZOWANY ZBIOR DANYCH (73 cechy)')
print('#'*80)
opt_dataset_summary = phase_summary_df[phase_summary_df['Zbior'] == 'Zoptymalizowany (73)'].groupby('Model').agg({
    'Dokladnosc': ['mean', 'std'],
    'Precyzja': ['mean', 'std'],
    'Czulosc': ['mean', 'std'],
    'F1': ['mean', 'std'],
    'ROC-AUC': ['mean', 'std'],
    'Bledne': 'sum'
}).round(4)

opt_dataset_flat = pd.DataFrame()
for model in opt_dataset_summary.index:
    opt_dataset_flat.loc[model, 'Dokladnosc (sr.)'] = opt_dataset_summary.loc[model, ('Dokladnosc', 'mean')]
    opt_dataset_flat.loc[model, 'Dokladnosc (std)'] = opt_dataset_summary.loc[model, ('Dokladnosc', 'std')]
    opt_dataset_flat.loc[model, 'Precyzja (sr.)'] = opt_dataset_summary.loc[model, ('Precyzja', 'mean')]
    opt_dataset_flat.loc[model, 'Precyzja (std)'] = opt_dataset_summary.loc[model, ('Precyzja', 'std')]
    opt_dataset_flat.loc[model, 'Czulosc (sr.)'] = opt_dataset_summary.loc[model, ('Czulosc', 'mean')]
    opt_dataset_flat.loc[model, 'Czulosc (std)'] = opt_dataset_summary.loc[model, ('Czulosc', 'std')]
    opt_dataset_flat.loc[model, 'F1 (sr.)'] = opt_dataset_summary.loc[model, ('F1', 'mean')]
    opt_dataset_flat.loc[model, 'F1 (std)'] = opt_dataset_summary.loc[model, ('F1', 'std')]
    opt_dataset_flat.loc[model, 'ROC-AUC (sr.)'] = opt_dataset_summary.loc[model, ('ROC-AUC', 'mean')]
    opt_dataset_flat.loc[model, 'ROC-AUC (std)'] = opt_dataset_summary.loc[model, ('ROC-AUC', 'std')]
    opt_dataset_flat.loc[model, 'Suma bledow'] = opt_dataset_summary.loc[model, ('Bledne', 'sum')]

opt_dataset_flat = opt_dataset_flat.sort_values('ROC-AUC (sr.)', ascending=False)
print(opt_dataset_flat.round(4))

print('\n' + '#'*80)
print('BEZPOSREDNIE POROWNANIE ZBIOROW DANYCH')
print('#'*80)
comparison_data = []
for model in full_dataset_flat.index:
    if model in opt_dataset_flat.index:
        comparison_data.append({
            'Model': model,
            'Pelny - ROC-AUC (sr.)': full_dataset_flat.loc[model, 'ROC-AUC (sr.)'],
            'Opt. - ROC-AUC (sr.)': opt_dataset_flat.loc[model, 'ROC-AUC (sr.)'],
            'Roznica ROC-AUC': opt_dataset_flat.loc[model, 'ROC-AUC (sr.)'] - full_dataset_flat.loc[model, 'ROC-AUC (sr.)'],
            'Pelny - F1 (sr.)': full_dataset_flat.loc[model, 'F1 (sr.)'],
            'Opt. - F1 (sr.)': opt_dataset_flat.loc[model, 'F1 (sr.)'],
            'Roznica F1': opt_dataset_flat.loc[model, 'F1 (sr.)'] - full_dataset_flat.loc[model, 'F1 (sr.)']
        })

comparison_df = pd.DataFrame(comparison_data).sort_values('Roznica ROC-AUC', ascending=False)
print(comparison_df.round(4))

full_summary_path = RESULTS_DIR / 'summary' / 'full_dataset_model_summary.csv'
opt_summary_path = RESULTS_DIR / 'summary' / 'optimized_dataset_model_summary.csv'
comparison_path = RESULTS_DIR / 'summary' / 'dataset_comparison.csv'

full_dataset_flat.to_csv(full_summary_path)
opt_dataset_flat.to_csv(opt_summary_path)
comparison_df.to_csv(comparison_path, index=False)

print(f'\nZapisano podsumowanie pelnego zbioru do: {full_summary_path}')
print(f'Zapisano podsumowanie zoptymalizowanego zbioru do: {opt_summary_path}')
print(f'Zapisano porownanie zbiorow do: {comparison_path}')

## 7. Wizualizacja - wpływ rozmiaru zbioru i rozkładu na wydajność


In [ ]:
phase_sizes = {
    'Faza 1: Mini Test': 2,
    'Faza 2: 1,000 probek': 1000,
    'Faza 2: 1,000 próbek': 1000,
    'Faza 3: 5,000 probek': 5000,
    'Faza 3: 5,000 próbek': 5000,
    'Faza 4: 10,000 probek': 10000,
    'Faza 4: 10,000 próbek': 10000,
    'Faza 5: 20,000 probek': 20000,
    'Faza 5: 20,000 próbek': 20000
}

phase_summary_df['Rozmiar'] = phase_summary_df['Faza'].map(phase_sizes)

dataset_types = ['Pelny (111)', 'Zoptymalizowany (73)']
dataset_colors = {
    'Pelny (111)': {'prefix': 'full', 'main_color': '#2c3e50'},
    'Zoptymalizowany (73)': {'prefix': 'optimized', 'main_color': '#8e44ad'}
}

model_colors = {
    'Regresja Logistyczna': '#e74c3c',
    'Random Forest': '#2ecc71',
    'XGBoost': '#3498db',
    'SVM': '#f39c12',
    'MLP': '#9b59b6'
}

for dataset_type in dataset_types:
    print(f'\n{"="*100}')
    print(f'WPLYW ROZMIARU ZBIORU TESTOWEGO - {dataset_type}')
    print(f'{"="*100}')

    fig, axes = plt.subplots(2, 2, figsize=(20, 14))
    dataset_title = 'Pelny zbior (111 cech)' if 'Pelny' in dataset_type else 'Zoptymalizowany zbior (73 cechy)'
    fig.suptitle(f'Wplyw rozmiaru zbioru testowego na metryki modeli - {dataset_title}',
                 fontsize=18, fontweight='bold')

    metrics_to_plot = ['ROC-AUC', 'F1', 'Dokladnosc', 'Precyzja']
    metric_names = ['ROC-AUC', 'F1', 'Dokladnosc', 'Precyzja']

    for idx, (metric_name, metric_col) in enumerate(zip(metric_names, metrics_to_plot)):
        ax = axes[idx // 2, idx % 2]

        for model_name in phase_summary_df['Model'].unique():
            model_data = phase_summary_df[
                (phase_summary_df['Model'] == model_name) &
                (phase_summary_df['Zbior'] == dataset_type) &
                (phase_summary_df['Rozklad'] == '50% phishing, 50% bezpieczne') &
                (phase_summary_df['Faza'] != 'Faza 1: Mini Test')
            ]

            if len(model_data) > 0:
                model_data_sorted = model_data.sort_values('Rozmiar')
                color = model_colors.get(model_name, '#34495e')
                ax.plot(model_data_sorted['Rozmiar'], model_data_sorted[metric_col],
                       marker='o', label=model_name, linewidth=2.5, markersize=8,
                       color=color, alpha=0.9)

        ax.set_xlabel('Rozmiar zbioru testowego', fontsize=12, fontweight='bold')
        ax.set_ylabel(metric_name, fontsize=12, fontweight='bold')
        ax.set_title(f'Zmiana {metric_name} wzgledem rozmiaru', fontweight='bold', fontsize=14)
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3)
        ax.set_xlim(left=0)
        ax.set_xticks([0, 2500, 5000, 7500, 10000, 12500, 15000, 17500, 20000])
        ax.set_xticklabels(['0', '2500', '5000', '7500', '10000', '12500', '15000', '17500', '20000'])

    plt.tight_layout()

    dataset_prefix = dataset_colors[dataset_type]['prefix']
    plt.savefig(PLOTS_DIR / 'summary' / f'size_impact_{dataset_prefix}_dataset.png',
                dpi=300, bbox_inches='tight')
    plt.show()

    print(f'Wizualizacja wplywu rozmiaru dla {dataset_type} zapisana')

print('Wszystkie wizualizacje wplywu rozmiaru zapisane')

## 8. Macierze pomyłek - wszystkie fazy


In [ ]:
phase_mapping = {
    'Faza 1: Mini Test': 1,
    'Faza 2: 1,000 probek': 2,
    'Faza 2: 1,000 próbek': 2,
    'Faza 3: 5,000 probek': 3,
    'Faza 3: 5,000 próbek': 3,
    'Faza 4: 10,000 probek': 4,
    'Faza 4: 10,000 próbek': 4,
    'Faza 5: 20,000 probek': 5,
    'Faza 5: 20,000 próbek': 5
}

dataset_types = ['full', 'optimized']
dataset_names = ['Pelny (111 cech)', 'Zoptymalizowany (73 cechy)']

for phase_name, phase_data in phase_results.items():
    phase_num = phase_mapping[phase_name]
    phase_folder = PLOTS_DIR / f'phase_{phase_num}'

    print(f'\n{"="*120}')
    print(f'MACIERZE POMYLEK - {phase_name.upper()}')
    print(f'{"="*120}')

    for dist_label in ['70% phishing, 30% bezpieczne', '30% phishing, 70% bezpieczne', '50% phishing, 50% bezpieczne']:

        for dataset_idx, (dataset_type, dataset_display_name) in enumerate(zip(dataset_types, dataset_names)):
            print(f'\n{"-"*80}')
            print(f'{dist_label} - {dataset_display_name}')
            print(f'{"-"*80}')

            models_with_data = []
            for model_name, datasets in phase_data.items():
                if dataset_type in datasets and dist_label in datasets[dataset_type]:
                    models_with_data.append(model_name)

            if not models_with_data:
                print(f'Brak danych dla {dataset_display_name}')
                continue

            n_models = len(models_with_data)
            n_cols = 3
            n_rows = (n_models + n_cols - 1) // n_cols

            fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 5))
            fig.suptitle(f'{phase_name} - {dist_label} - {dataset_display_name}',
                        fontsize=18, fontweight='bold')

            if n_models == 1:
                axes = np.array([axes])
            axes = axes.flatten()

            for idx, model_name in enumerate(models_with_data):
                ax = axes[idx]
                dist_data = phase_data[model_name][dataset_type][dist_label]
                # Fixed: confusion_matrix is inside metrics dictionary
                cm = dist_data['metrics']['confusion_matrix']

                # Color based on dataset type
                cmap = 'Blues' if dataset_type == 'full' else 'Purples'

                sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                           cbar_kws={'label': 'Liczba probek'},
                           annot_kws={'fontsize': 11, 'fontweight': 'bold'})

                ax.set_title(f'{model_name}\nROC-AUC: {dist_data["metrics"]["roc_auc"]:.4f} | '
                           f'F1: {dist_data["metrics"]["f1"]:.4f}',
                           fontweight='bold', fontsize=12)
                ax.set_ylabel('Rzeczywista klasa', fontsize=11)
                ax.set_xlabel('Przewidziana klasa', fontsize=11)
                ax.set_xticklabels(['Bezpieczne', 'Phishing'])
                ax.set_yticklabels(['Bezpieczne', 'Phishing'])

            for idx in range(n_models, len(axes)):
                fig.delaxes(axes[idx])

            plt.tight_layout()

            safe_dist_label = dist_label.replace('/', '-').replace(' ', '_').replace(',', '')
            filename = f'confusion_matrices_{safe_dist_label}_{dataset_type}.png'
            plt.savefig(phase_folder / filename, dpi=300, bbox_inches='tight')
            plt.show()

            print(f'Zapisano macierze pomylek do: {phase_folder / filename}')

print('\nWszystkie macierze pomylek wygenerowane i zapisane')

## 9. Porównanie metryk - wszystkie fazy


In [ ]:
print('='*100)
print('ŚREDNIE STATYSTYKI WSZYSTKICH FAZ')
print('='*100)

avg_metrics_data = []

for phase_name, phase_data in phase_results.items():
    if phase_name == 'Faza 1: Mini Test':
        continue

    for dist_label in ['70% phishing, 30% bezpieczne', '30% phishing, 70% bezpieczne', '50% phishing, 50% bezpieczne']:
        for model_name, datasets in phase_data.items():
            for dataset_type in ['full', 'optimized']:
                if dataset_type in datasets and dist_label in datasets[dataset_type]:
                    metrics = datasets[dataset_type][dist_label]['metrics']
                    avg_metrics_data.append({
                        'model': model_name,
                        'dataset': dataset_type,
                        'distribution': dist_label,
                        'accuracy': metrics['accuracy'],
                        'precision': metrics['precision'],
                        'recall': metrics['recall'],
                        'f1': metrics['f1'],
                        'roc_auc': metrics['roc_auc']
                    })

avg_df = pd.DataFrame(avg_metrics_data)

model_averages = avg_df.groupby(['model', 'dataset']).agg({
    'accuracy': 'mean',
    'precision': 'mean',
    'recall': 'mean',
    'f1': 'mean',
    'roc_auc': 'mean'
}).reset_index()

print('\nŚrednie metryki według modelu i typu zbioru (fazy 2-5):')
print(model_averages.round(4))

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Średnie statystyki modeli (fazy 2-5)', fontsize=18, fontweight='bold')

metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
metric_names = ['Dokładność', 'Precyzja', 'Czułość', 'F1', 'ROC-AUC']

model_colors = {
    'Regresja Logistyczna': '#e74c3c',
    'Random Forest': '#2ecc71',
    'XGBoost': '#3498db',
    'SVM': '#f39c12',
    'MLP': '#9b59b6'
}

for idx, (metric_key, metric_name) in enumerate(zip(metrics_to_plot, metric_names)):
    if idx < 5:
        ax = axes[idx // 3, idx % 3]

        full_data = model_averages[model_averages['dataset'] == 'full'].sort_values(metric_key, ascending=False)
        opt_data = model_averages[model_averages['dataset'] == 'optimized'].sort_values(metric_key, ascending=False)

        x = np.arange(len(full_data))
        width = 0.35

        full_dict = dict(zip(full_data['model'], full_data[metric_key]))
        opt_dict = dict(zip(opt_data['model'], opt_data[metric_key]))

        model_avg_scores = {}
        for model in full_dict.keys():
            if model in opt_dict:
                model_avg_scores[model] = (full_dict[model] + opt_dict[model]) / 2
            else:
                model_avg_scores[model] = full_dict[model]

        sorted_models = sorted(model_avg_scores.keys(), key=lambda x: model_avg_scores[x], reverse=True)

        full_values = [full_dict.get(model, 0) for model in sorted_models]
        opt_values = [opt_dict.get(model, 0) for model in sorted_models]

        bars1 = ax.bar(x - width/2, full_values, width, label='Pełny zbiór (111)', alpha=0.8, color='#3498db')
        bars2 = ax.bar(x + width/2, opt_values, width, label='Zoptymalizowany zbiór (73)', alpha=0.8, color='#e67e22')

        ax.set_xlabel('Model', fontsize=11, fontweight='bold')
        ax.set_ylabel(metric_name, fontsize=11, fontweight='bold')
        ax.set_title(f'Średnia {metric_name}', fontweight='bold', fontsize=12)
        ax.set_xticks(x)
        ax.set_xticklabels(sorted_models, rotation=45, ha='right', fontsize=9)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3, axis='y')
        ax.set_ylim([0, 1.05])

        for bars in [bars1, bars2]:
            for bar in bars:
                height = bar.get_height()
                if height > 0:
                    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                           f'{height:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

fig.delaxes(axes[1, 2])

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'summary' / 'average_metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print('\n' + '='*100)
print('RANKING MODELI WEDŁUG ŚREDNICH WYNIKÓW (WSZYSTKIE METRYKI)')
print('='*100)

weights = {
    'roc_auc': 0.3,
    'f1': 0.25,
    'accuracy': 0.2,
    'precision': 0.15,
    'recall': 0.1
}

model_final_scores = []
for _, row in model_averages.iterrows():
    weighted_score = sum(row[metric] * weight for metric, weight in weights.items())
    model_final_scores.append({
        'Model': row['model'],
        'Zbiór': 'Pełny (111)' if row['dataset'] == 'full' else 'Zoptymalizowany (73)',
        'Wynik ważony': weighted_score,
        'ROC-AUC': row['roc_auc'],
        'F1': row['f1'],
        'Dokładność': row['accuracy'],
        'Precyzja': row['precision'],
        'Czułość': row['recall']
    })

final_ranking = pd.DataFrame(model_final_scores).sort_values('Wynik ważony', ascending=False)
print(final_ranking.round(4))

final_ranking.to_csv(RESULTS_DIR / 'summary' / 'average_model_performance.csv', index=False)

print(f'\n✅ Wizualizacja średnich statystyk zapisana do: average_metrics_comparison.png')
print(f'✅ Ranking średnich wyników zapisany do: average_model_performance.csv')
print('✅ Analiza średnich statystyk zakończona')

## 10. Analiza błędnie sklasyfikowanych próbek (Faza 5: rozkład 50/50)


In [ ]:
def calculate_confidence_error(_, predicted_label, probability):
    if predicted_label == 1:
        confidence = probability
    else:
        confidence = 1 - probability
    return confidence

def create_detailed_misclassification_dataframe(model_name, dataset_type, misclassified, X_test):
    if not misclassified:
        return pd.DataFrame()

    detailed_records = []

    for miss in misclassified:
        confidence = calculate_confidence_error(
            miss['true_label'],
            miss['predicted_label'],
            miss['probability']
        )

        record = {
            'Model': model_name,
            'Zbiór': 'Pełny (111)' if dataset_type == 'full' else 'Zoptymalizowany (73)',
            'Indeks próbki': miss['index'],
            'Prawdziwa etykieta': 'Phishing' if miss['true_label'] == 1 else 'Bezpieczne',
            'Przewidziana etykieta': 'Phishing' if miss['predicted_label'] == 1 else 'Bezpieczne',
            'Prawdopodobieństwo': miss['probability'],
            'Pewność błędu': confidence
        }

        top_features = sorted(
            [(k, v) for k, v in miss['features'].items()],
            key=lambda x: abs(x[1]),
            reverse=True
        )[:10]

        for i, (feat_name, feat_value) in enumerate(top_features, 1):
            readable_name = get_readable_name(feat_name)
            record[f'Cecha {i}'] = readable_name
            record[f'Wartość {i}'] = feat_value

        detailed_records.append(record)

    df = pd.DataFrame(detailed_records)
    df = df.sort_values('Pewność błędu', ascending=False)
    return df

def print_top_misclassifications(df, model_name, dataset_type, top_n=5):
    print(f'\n{"="*100}')
    print(f'TOP {top_n} NAJBARDZIEJ PEWNYCH BŁĘDÓW: {model_name} - {dataset_type.upper()}')
    print(f'{"="*100}')
    print(f'Całkowita liczba błędnych klasyfikacji: {len(df)}')
    print(f'{"-"*100}\n')

    for i, row in df.head(top_n).iterrows():
        print(f'#{i+1}. Próbka indeks: {int(row["Indeks próbki"])}')
        print(f'   Prawdziwa: {row["Prawdziwa etykieta"]} → Przewidziana: {row["Przewidziana etykieta"]}')
        print(f'   Pewność błędu: {row["Pewność błędu"]:.4f} (Prawdopodobieństwo: {row["Prawdopodobieństwo"]:.4f})')
        print(f'   Top 3 cechy:')
        for j in range(1, 4):
            if f'Cecha {j}' in row:
                print(f'     • {row[f"Cecha {j}"]}: {row[f"Wartość {j}"]:.4f}')
        print()

all_misclassifications = []

print('='*100)
print('ANALIZA BŁĘDÓW - WSZYSTKIE FAZY (ROZKŁAD 50/50)')
print('='*100)

phase_mapping = {
    'Faza 1: Mini Test': 1,
    'Faza 2: 1,000 próbek': 2,
    'Faza 3: 5,000 próbek': 3,
    'Faza 4: 10,000 próbek': 4,
    'Faza 5: 20,000 próbek': 5
}

for phase_name, phase_data in phase_results.items():
    phase_num = phase_mapping[phase_name]
    phase_misclass = []

    print(f'\n{"="*100}')
    print(f'{phase_name.upper()}')
    print(f'{"="*100}')

    for model_name, datasets in phase_data.items():
        for dataset_type, distributions in datasets.items():
            dist_50_50 = distributions.get('50% phishing, 50% bezpieczne')

            if dist_50_50 and dist_50_50['misclassified']:
                test_df = test_datasets[dataset_type]
                X_test = test_df.iloc[:, :-1]

                df_misclass = create_detailed_misclassification_dataframe(
                    model_name, dataset_type,
                    dist_50_50['misclassified'],
                    X_test
                )

                phase_misclass.append(df_misclass)
                all_misclassifications.append(df_misclass)
                print_top_misclassifications(df_misclass, model_name, dataset_type, top_n=3)

    if phase_misclass:
        combined_phase = pd.concat(phase_misclass, ignore_index=True)
        combined_phase = combined_phase.sort_values('Pewność błędu', ascending=False)

        csv_path = RESULTS_DIR / f'phase_{phase_num}' / f'phase_{phase_num}_misclassifications.csv'
        combined_phase.to_csv(csv_path, index=False)

        print(f'\n✅ Zapisano {len(combined_phase):,} błędnych klasyfikacji do: {csv_path}')

if all_misclassifications:
    combined_df = pd.concat(all_misclassifications, ignore_index=True)
    combined_df = combined_df.sort_values('Pewność błędu', ascending=False)

    csv_path = RESULTS_DIR / 'summary' / 'all_phases_misclassifications.csv'
    combined_df.to_csv(csv_path, index=False)

    print(f'\n{"="*100}')
    print(f'✅ Zapisano {len(combined_df):,} błędnych klasyfikacji (wszystkie fazy) do: {csv_path}')
    print(f'{"="*100}')

    print(f'\nPodsumowanie błędów według modelu:')
    summary = combined_df.groupby(['Model', 'Zbiór']).agg({
        'Pewność błędu': ['count', 'mean', 'max']
    }).round(4)
    print(summary)

print('\n✅ Analiza błędnych klasyfikacji zakończona')

## 11. Rozkład błędnych klasyfikacji - wszystkie fazy


In [ ]:
phase_mapping = {
    'Faza 1: Mini Test': 1,
    'Faza 2: 1,000 próbek': 2,
    'Faza 3: 5,000 próbek': 3,
    'Faza 4: 10,000 próbek': 4,
    'Faza 5: 20,000 próbek': 5
}

for phase_name, phase_data in phase_results.items():
    phase_num = phase_mapping[phase_name]
    phase_folder = PLOTS_DIR / f'phase_{phase_num}'

    print(f'\n{"="*120}')
    print(f'ROZKLAD BLEDNYCH KLASYFIKACJI - {phase_name.upper()}')
    print(f'{"="*120}')

    phase_size = phase_summary_df[phase_summary_df['Faza'] == phase_name]['Rozmiar'].iloc[0]

    for dist_label in ['70% phishing, 30% bezpieczne', '30% phishing, 70% bezpieczne', '50% phishing, 50% bezpieczne']:
        has_data = any(
            dataset_type in datasets and dist_label in datasets[dataset_type]
            for datasets in phase_data.values()
            for dataset_type in ['full', 'optimized']
        )

        if not has_data:
            continue

        fig, axes = plt.subplots(3, 1, figsize=(16, 18))
        fig.suptitle(f'Rozklad blednych klasyfikacji - {phase_name} - {dist_label}',
                    fontsize=18, fontweight='bold')

        dataset_types = ['full', 'optimized']
        dataset_titles = ['Pelny zbior (111 cech)', 'Zoptymalizowany zbior (73 cechy)']
        dataset_colors = ['#3498db', '#9b59b6']

        for idx, (dataset_type, title, color) in enumerate(zip(dataset_types, dataset_titles, dataset_colors)):
            ax = axes[idx]

            model_names = []
            correct_counts = []
            incorrect_counts = []

            for model_name, datasets in phase_data.items():
                if dataset_type in datasets and dist_label in datasets[dataset_type]:
                    dist_data = datasets[dataset_type][dist_label]

                    model_names.append(model_name)
                    incorrect = len(dist_data['misclassified'])
                    total = phase_size
                    correct = total - incorrect

                    correct_counts.append(correct)
                    incorrect_counts.append(incorrect)

            if not model_names:
                ax.text(0.5, 0.5, f'Brak danych dla {title}',
                       ha='center', va='center', transform=ax.transAxes, fontsize=14)
                ax.set_title(title, fontweight='bold', fontsize=14)
                continue

            x = np.arange(len(model_names))
            width = 0.6

            bars1 = ax.bar(x, correct_counts, width, label='Poprawne', color='#2ecc71', alpha=0.8)
            bars2 = ax.bar(x, incorrect_counts, width, bottom=correct_counts,
                           label='Bledne', color='#e74c3c', alpha=0.8)

            ax.set_xlabel('Model', fontsize=12, fontweight='bold')
            ax.set_ylabel('Liczba probek', fontsize=12, fontweight='bold')
            ax.set_title(f'{title} - Rozklad klasyfikacji', fontweight='bold', fontsize=14)
            ax.set_xticks(x)
            ax.set_xticklabels(model_names, rotation=45, ha='right')
            ax.legend()
            ax.grid(True, alpha=0.3, axis='y')

            for i, (correct, incorrect) in enumerate(zip(correct_counts, incorrect_counts)):
                total = correct + incorrect
                y_offset = max(total * 0.02, 1)
                ax.text(i, total + y_offset, f'{correct}/{total}',
                       ha='center', va='bottom', fontsize=10, fontweight='bold')

                if incorrect > 0:
                    ax.text(i, correct + incorrect/2, f'{incorrect}',
                           ha='center', va='center', fontsize=10, fontweight='bold', color='white')

        ax = axes[2]

        all_models = set()
        dataset_data = {'full': {}, 'optimized': {}}

        for dataset_type in dataset_types:
            for model_name, datasets in phase_data.items():
                if dataset_type in datasets and dist_label in datasets[dataset_type]:
                    all_models.add(model_name)
                    dist_data = datasets[dataset_type][dist_label]
                    incorrect = len(dist_data['misclassified'])
                    dataset_data[dataset_type][model_name] = incorrect

        if all_models:
            model_names = sorted(list(all_models))
            x = np.arange(len(model_names))
            width = 0.35

            full_errors = [dataset_data['full'].get(model, 0) for model in model_names]
            opt_errors = [dataset_data['optimized'].get(model, 0) for model in model_names]

            bars1 = ax.bar(x - width/2, full_errors, width, label='Pelny zbior (111)',
                          color='#4A90E2', alpha=0.8)
            bars2 = ax.bar(x + width/2, opt_errors, width, label='Zoptymalizowany zbior (73)',
                          color='#e67e22', alpha=0.8)

            ax.set_xlabel('Model', fontsize=12, fontweight='bold')
            ax.set_ylabel('Liczba blednych klasyfikacji', fontsize=12, fontweight='bold')
            ax.set_title('Porownanie blednych klasyfikacji - Pelny vs Zoptymalizowany',
                        fontweight='bold', fontsize=14)
            ax.set_xticks(x)
            ax.set_xticklabels(model_names, rotation=45, ha='right')
            ax.legend()
            ax.grid(True, alpha=0.3, axis='y')

            max_error = 0
            if full_errors or opt_errors:
                max_error = max(max(full_errors, default=0), max(opt_errors, default=0))

            ax.set_ylim(top=max_error * 1.15)

            for bars, values in [(bars1, full_errors), (bars2, opt_errors)]:
                for bar, val in zip(bars, values):
                    if val > 0:
                        height = bar.get_height()
                        ax.text(bar.get_x() + bar.get_width()/2., height,
                               f'{val}', ha='center', va='bottom', fontsize=9, fontweight='bold')

        plt.tight_layout()
        safe_dist_label = dist_label.replace('%', 'pct').replace(',', '').replace(' ', '_')
        plt.savefig(phase_folder / f'misclassification_dist_separated_{safe_dist_label}.png',
                   dpi=300, bbox_inches='tight')
        plt.show()

        print(f'Zapisano rozklad bledow z rozdzieleniem dla: {dist_label}')

print(f'\n{"="*120}')
print('POROWNANIE ROZKLADOW BLEDOW - WSZYSTKIE FAZY (ROZKLAD 50/50)')
print(f'{"="*120}')

comparison_data = {'full': {}, 'optimized': {}}
phase_names = []

for phase_name, phase_data in phase_results.items():
    if phase_name == 'Faza 1: Mini Test':
        continue

    phase_names.append(phase_name)
    dist_label = '50% phishing, 50% bezpieczne'

    for dataset_type in ['full', 'optimized']:
        if phase_name not in comparison_data[dataset_type]:
            comparison_data[dataset_type][phase_name] = {}

        for model_name, datasets in phase_data.items():
            if dataset_type in datasets and dist_label in datasets[dataset_type]:
                dist_data = datasets[dataset_type][dist_label]
                errors = len(dist_data['misclassified'])
                phase_size = phase_sizes[phase_name]
                error_rate = (errors / phase_size) * 100
                comparison_data[dataset_type][phase_name][model_name] = error_rate

if phase_names:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle('Wspolczynnik bledow wedlug rozmiaru fazy - Pelny vs Zoptymalizowany zbior (50/50)',
                 fontsize=16, fontweight='bold')

    all_models = set()
    for dataset_type in ['full', 'optimized']:
        for phase_data in comparison_data[dataset_type].values():
            all_models.update(phase_data.keys())

    model_colors = {
        'Regresja Logistyczna': '#e74c3c',
        'Random Forest': '#2ecc71',
        'XGBoost': '#3498db',
        'SVM': '#f39c12',
        'MLP': '#9b59b6'
    }

    for ax_idx, (dataset_type, title) in enumerate([('full', 'Pelny zbior (111 cech)'),
                                                   ('optimized', 'Zoptymalizowany zbior (73 cechy)')]):
        ax = ax1 if ax_idx == 0 else ax2

        for model_name in all_models:
            phase_sizes_list = []
            error_rates = []

            for phase_name in phase_names:
                if (phase_name in comparison_data[dataset_type] and
                    model_name in comparison_data[dataset_type][phase_name]):
                    phase_sizes_list.append(phase_sizes[phase_name])
                    error_rates.append(comparison_data[dataset_type][phase_name][model_name])

            if phase_sizes_list:
                color = model_colors.get(model_name, '#34495e')
                ax.plot(phase_sizes_list, error_rates, marker='o', label=model_name,
                       linewidth=2.5, markersize=8, color=color, alpha=0.9)

        ax.set_xlabel('Rozmiar zbioru testowego', fontsize=12, fontweight='bold')
        ax.set_ylabel('Wspolczynnik bledow (%)', fontsize=12, fontweight='bold')
        ax.set_title(title, fontweight='bold', fontsize=14)
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3)
        ax.set_xlim(left=0)

    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'summary' / 'error_rates_comparison_phases.png', dpi=300, bbox_inches='tight')
    plt.show()

print('Wszystkie rozklady blednych klasyfikacji z rozdzieleniem zbiorow zapisane')
print('Porownanie wspolczynnikow bledow wedlug faz zapisane')

## 12. Ranking modeli (wszystkie fazy)


In [ ]:
def calculate_ranking_score(metrics):
    weights = {
        'roc_auc': 0.3,
        'f1': 0.25,
        'accuracy': 0.2,
        'precision': 0.15,
        'recall': 0.1
    }
    score = sum(metrics[key] * weight for key, weight in weights.items())
    return score

ranking_data = []

for phase_name, models in phase_results.items():
    for model_name, datasets in models.items():
        for dataset_type, distributions in datasets.items():
            for dist_label, results in distributions.items():
                metrics = results['metrics']
                score = calculate_ranking_score(metrics)

                ranking_data.append({
                    'Faza': phase_name,
                    'Model': model_name,
                    'Zbiór': 'Pełny (111)' if dataset_type == 'full' else 'Zoptymalizowany (73)',
                    'Rozkład': dist_label,
                    'Wynik rankingowy': score,
                    'ROC-AUC': metrics['roc_auc'],
                    'F1': metrics['f1'],
                    'Dokładność': metrics['accuracy'],
                    'Błędne': len(results['misclassified'])
                })

ranking_df = pd.DataFrame(ranking_data)
ranking_df = ranking_df.sort_values('Wynik rankingowy', ascending=False)

print('='*100)
print('RANKING MODELI - TOP 10 (WSZYSTKIE FAZY)')
print('='*100)
print(ranking_df.head(10).to_string(index=False))

ranking_df.to_csv(RESULTS_DIR / 'summary' / 'complete_ranking.csv', index=False)
print(f'\n✅ Ranking zapisany do: {RESULTS_DIR / "summary" / "complete_ranking.csv"}')

phase_mapping = {
    'Faza 1: Mini Test': 1,
    'Faza 2: 1,000 próbek': 2,
    'Faza 3: 5,000 próbek': 3,
    'Faza 4: 10,000 próbek': 4,
    'Faza 5: 20,000 próbek': 5
}

for phase_name in ranking_df['Faza'].unique():
    phase_num = phase_mapping[phase_name]
    phase_ranking = ranking_df[ranking_df['Faza'] == phase_name].copy()

    csv_path = RESULTS_DIR / f'phase_{phase_num}' / f'phase_{phase_num}_ranking.csv'
    phase_ranking.to_csv(csv_path, index=False)

    print(f'\n{"="*100}')
    print(f'RANKING - {phase_name.upper()}')
    print(f'{"="*100}')
    print(phase_ranking.head(10).to_string(index=False))
    print(f'\n✅ Zapisano do: {csv_path}')

print('\n' + '='*100)
print('RANKING MODELI - TOP 10 DLA KAŻDEJ FAZY (ROZKŁAD 50/50)')
print('='*100)
for phase_name in phase_summary_df['Faza'].unique():
    phase_50_50 = ranking_df[
        (ranking_df['Faza'] == phase_name) &
        (ranking_df['Rozkład'] == '50% phishing, 50% bezpieczne')
    ].sort_values('Wynik rankingowy', ascending=False)

    print(f'\n{phase_name}:')
    print(phase_50_50.head(10).to_string(index=False))
    print()

## 13. Szczegółowe statystyki dla wszystkich faz


In [ ]:
print('='*120)
print('SZCZEGOLOWE STATYSTYKI WSZYSTKICH FAZ')
print('='*120)

phase_mapping = {
    'Faza 1: Mini Test': 1,
    'Faza 2: 1,000 próbek': 2,
    'Faza 3: 5,000 próbek': 3,
    'Faza 4: 10,000 próbek': 4,
    'Faza 5: 20,000 próbek': 5
}

dataset_types = ['Pelny (111)', 'Zoptymalizowany (73)']
dataset_colors = {'Pelny (111)': '#3498db', 'Zoptymalizowany (73)': '#9b59b6'}

for phase_name in phase_summary_df['Faza'].unique():
    phase_num = phase_mapping[phase_name]
    phase_folder = PLOTS_DIR / f'phase_{phase_num}'

    print(f"\n{'='*120}")
    print(f'{phase_name.upper()}')
    print(f"{'='*120}")

    for dataset_type in dataset_types:
        dataset_color = dataset_colors[dataset_type]
        phase_data = phase_summary_df[
            (phase_summary_df['Faza'] == phase_name) &
            (phase_summary_df['Zbior'] == dataset_type)
        ]

        if phase_data.empty:
            print(f"\nBrak danych dla {dataset_type} w {phase_name}")
            continue

        print(f"\n{'#'*80}")
        print(f'{dataset_type} - {phase_name}')
        print(f"{'#'*80}")

        for dist_label in phase_data['Rozklad'].unique():
            dist_data = phase_data[phase_data['Rozklad'] == dist_label]

            print(f"\n{dist_label}:")
            print(f"{'-'*100}")

            dist_data_sorted = dist_data.sort_values('ROC-AUC', ascending=False)
            display_cols = ['Model', 'Dokladnosc', 'Precyzja', 'Czulosc', 'F1', 'ROC-AUC', 'Bledne']
            print(dist_data_sorted[display_cols].to_string(index=False))

            print(f"\nStatystyki dla {dist_label} - {dataset_type}:")
            stats = dist_data_sorted[['Dokladnosc', 'Precyzja', 'Czulosc', 'F1', 'ROC-AUC']].describe().loc[['mean', 'std', 'min', 'max']]
            print(stats.round(4))

    fig, axes = plt.subplots(3, 2, figsize=(24, 18))
    fig.suptitle(f'Porownanie statystyk - {phase_name} - Pelny vs Zoptymalizowany',
                 fontsize=18, fontweight='bold')

    full_data = phase_summary_df[
        (phase_summary_df['Faza'] == phase_name) &
        (phase_summary_df['Zbior'] == 'Pelny (111)')
    ]
    opt_data = phase_summary_df[
        (phase_summary_df['Faza'] == phase_name) &
        (phase_summary_df['Zbior'] == 'Zoptymalizowany (73)')
    ]

    ax_full_top = axes[0, 0]
    if not full_data.empty:
        top_models = full_data.nlargest(10, 'ROC-AUC')
        top_models = top_models.reset_index(drop=True)
        top_models['Label'] = top_models['Model'] + '\n(' + top_models['Rozklad'].str[:15] + '...)'
        colors = plt.cm.Blues(np.linspace(0.4, 1, len(top_models)))
        bars = ax_full_top.barh(range(len(top_models)), top_models['ROC-AUC'], color=colors, alpha=0.9)
        ax_full_top.set_yticks(range(len(top_models)))
        ax_full_top.set_yticklabels(top_models['Label'].values, fontsize=9)
        ax_full_top.set_xlabel('ROC-AUC', fontsize=11, fontweight='bold')
        ax_full_top.set_title('TOP 10 ROC-AUC - Pelny (111)', fontweight='bold', fontsize=12)
        ax_full_top.invert_yaxis()
        ax_full_top.grid(True, alpha=0.3, axis='x')
        for i, val in enumerate(top_models['ROC-AUC']):
            ax_full_top.text(val + 0.001, i, f'{val:.3f}', va='center', fontsize=8, fontweight='bold')

    ax_opt_top = axes[0, 1]
    if not opt_data.empty:
        top_models = opt_data.nlargest(10, 'ROC-AUC')
        top_models = top_models.reset_index(drop=True)
        top_models['Label'] = top_models['Model'] + '\n(' + top_models['Rozklad'].str[:15] + '...)'
        colors = plt.cm.Purples(np.linspace(0.4, 1, len(top_models)))
        bars = ax_opt_top.barh(range(len(top_models)), top_models['ROC-AUC'], color=colors, alpha=0.9)
        ax_opt_top.set_yticks(range(len(top_models)))
        ax_opt_top.set_yticklabels(top_models['Label'].values, fontsize=9)
        ax_opt_top.set_xlabel('ROC-AUC', fontsize=11, fontweight='bold')
        ax_opt_top.set_title('TOP 10 ROC-AUC - Zoptymalizowany (73)', fontweight='bold', fontsize=12)
        ax_opt_top.invert_yaxis()
        ax_opt_top.grid(True, alpha=0.3, axis='x')
        for i, val in enumerate(top_models['ROC-AUC']):
            ax_opt_top.text(val + 0.001, i, f'{val:.3f}', va='center', fontsize=8, fontweight='bold')

    ax_full_err = axes[1, 0]
    if not full_data.empty:
        error_summary = full_data.groupby('Model')['Bledne'].sum().sort_values(ascending=True)
        colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(error_summary)))
        bars = ax_full_err.barh(range(len(error_summary)), error_summary.values, color=colors, alpha=0.9)
        ax_full_err.set_yticks(range(len(error_summary)))
        ax_full_err.set_yticklabels(error_summary.index, fontsize=10)
        ax_full_err.set_xlabel('Suma bledow', fontsize=11, fontweight='bold')
        ax_full_err.set_title('Bledy - Pelny (111)', fontweight='bold', fontsize=12)
        ax_full_err.invert_yaxis()
        ax_full_err.grid(True, alpha=0.3, axis='x')
        max_err = error_summary.max() if len(error_summary) else 0
        ax_full_err.set_xlim(0, max_err * 1.15 if max_err > 0 else 1)
        for i, val in enumerate(error_summary.values):
            ax_full_err.text(val + max_err * 0.01 if max_err>0 else val + 0.1, i, f'{int(val)}', va='center', fontsize=9, fontweight='bold')

    ax_opt_err = axes[1, 1]
    if not opt_data.empty:
        error_summary = opt_data.groupby('Model')['Bledne'].sum().sort_values(ascending=True)
        colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(error_summary)))
        bars = ax_opt_err.barh(range(len(error_summary)), error_summary.values, color=colors, alpha=0.9)
        ax_opt_err.set_yticks(range(len(error_summary)))
        ax_opt_err.set_yticklabels(error_summary.index, fontsize=10)
        ax_opt_err.set_xlabel('Suma bledow', fontsize=11, fontweight='bold')
        ax_opt_err.set_title('Bledy - Zoptymalizowany (73)', fontweight='bold', fontsize=12)
        ax_opt_err.invert_yaxis()
        ax_opt_err.grid(True, alpha=0.3, axis='x')
        max_err = error_summary.max() if len(error_summary) else 0
        ax_opt_err.set_xlim(0, max_err * 1.15 if max_err > 0 else 1)
        for i, val in enumerate(error_summary.values):
            ax_opt_err.text(val + max_err * 0.01 if max_err>0 else val + 0.1, i, f'{int(val)}', va='center', fontsize=9, fontweight='bold')

    metrics = ['ROC-AUC', 'F1', 'Dokladnosc']
    width = 0.22

    ax_full_metrics = axes[2, 0]
    if not full_data.empty:
        full_avg = full_data.groupby('Model')[metrics].mean().sort_values('ROC-AUC', ascending=False)
        top_n = min(8, len(full_avg))
        models = full_avg.index[:top_n].tolist()
        x = np.arange(len(models))

        for i, metric in enumerate(metrics):
            vals = [full_avg.loc[m, metric] for m in models]
            ax_full_metrics.bar(x + (i - 1) * width, vals, width, label=metric, alpha=0.9)

        ax_full_metrics.set_xlabel('Model', fontsize=11, fontweight='bold')
        ax_full_metrics.set_ylabel('Wartosc metryki', fontsize=11, fontweight='bold')
        ax_full_metrics.set_title('Wszystkie metryki - Pelny', fontweight='bold', fontsize=12)
        ax_full_metrics.set_xticks(x)
        ax_full_metrics.set_xticklabels(models, rotation=45, ha='right', fontsize=9)
        ax_full_metrics.legend(fontsize=8)
        ax_full_metrics.grid(True, alpha=0.3, axis='y')
        ax_full_metrics.set_ylim([0, 1.05])

        for rect in ax_full_metrics.patches:
            height = rect.get_height()
            if height > 0:
                ax_full_metrics.text(rect.get_x() + rect.get_width()/2., height + 0.01, f'{height:.3f}', ha='center', va='bottom', fontsize=8)

    ax_opt_metrics = axes[2, 1]
    if not opt_data.empty:
        opt_avg = opt_data.groupby('Model')[metrics].mean().sort_values('ROC-AUC', ascending=False)
        top_n = min(8, len(opt_avg))
        models = opt_avg.index[:top_n].tolist()
        x = np.arange(len(models))

        for i, metric in enumerate(metrics):
            vals = [opt_avg.loc[m, metric] for m in models]
            ax_opt_metrics.bar(x + (i - 1) * width, vals, width, label=metric, alpha=0.9)

        ax_opt_metrics.set_xlabel('Model', fontsize=11, fontweight='bold')
        ax_opt_metrics.set_ylabel('Wartosc metryki', fontsize=11, fontweight='bold')
        ax_opt_metrics.set_title('Wszystkie metryki - Zoptymalizowany', fontweight='bold', fontsize=12)
        ax_opt_metrics.set_xticks(x)
        ax_opt_metrics.set_xticklabels(models, rotation=45, ha='right', fontsize=9)
        ax_opt_metrics.legend(fontsize=8)
        ax_opt_metrics.grid(True, alpha=0.3, axis='y')
        ax_opt_metrics.set_ylim([0, 1.05])

        for rect in ax_opt_metrics.patches:
            height = rect.get_height()
            if height > 0:
                ax_opt_metrics.text(rect.get_x() + rect.get_width()/2., height + 0.01, f'{height:.3f}', ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    plt.savefig(phase_folder / f'phase_{phase_num}_statistics_separated.png', dpi=300, bbox_inches='tight')
    plt.show()

    print(f"\nZapisano statystyki z rozdzieleniem do: {phase_folder / f'phase_{phase_num}_statistics_separated.png'}")

print(f"\n{'='*120}")
print('PODSUMOWANIE WSZYSTKICH FAZ - POROWNANIE ZBIOROW')
print(f"{'='*120}")

full_summary = phase_summary_df[phase_summary_df['Zbior'] == 'Pelny (111)'].groupby('Model').agg({
    'ROC-AUC': ['mean', 'std', 'count'],
    'F1': ['mean', 'std'],
    'Dokladnosc': ['mean', 'std'],
    'Bledne': 'sum'
}).round(4)

opt_summary = phase_summary_df[phase_summary_df['Zbior'] == 'Zoptymalizowany (73)'].groupby('Model').agg({
    'ROC-AUC': ['mean', 'std', 'count'],
    'F1': ['mean', 'std'],
    'Dokladnosc': ['mean', 'std'],
    'Bledne': 'sum'
}).round(4)

print('\nPORÓWNANIE SREDNIEJ WYDAJNOSCI MODELI:')
print('='*60)
print('\nPELNY ZBIOR (111 cech):')
print(full_summary)
print('\nZOPTYMALIZOWANY ZBIOR (73 cechy):')
print(opt_summary)

print('\nPROCENTOWA POPRAWA ZOPTYMALIZOWANEGO ZBIORU:')
print('='*60)
improvement_data = []
common_models = set(full_summary.index) & set(opt_summary.index)
for model in common_models:
    full_roc = full_summary.loc[model, ('ROC-AUC', 'mean')]
    opt_roc = opt_summary.loc[model, ('ROC-AUC', 'mean')]
    roc_improvement = ((opt_roc - full_roc) / full_roc) * 100

    full_f1 = full_summary.loc[model, ('F1', 'mean')]
    opt_f1 = opt_summary.loc[model, ('F1', 'mean')]
    f1_improvement = ((opt_f1 - full_f1) / full_f1) * 100

    improvement_data.append({
        'Model': model,
        'ROC-AUC Poprawa (%)': roc_improvement,
        'F1 Poprawa (%)': f1_improvement,
        'Pelny ROC-AUC': full_roc,
        'Zoptym ROC-AUC': opt_roc
    })

improvement_df = pd.DataFrame(improvement_data).sort_values('ROC-AUC Poprawa (%)', ascending=False)
print(improvement_df.round(3))

improvement_path = RESULTS_DIR / 'summary' / 'dataset_improvement_analysis.csv'
improvement_df.to_csv(improvement_path, index=False)


## 14. Podsumowanie końcowe - testowanie fazowe


In [ ]:
print('='*100)
print('PODSUMOWANIE KONCOWE - TESTOWANIE FAZOWE MODELI')
print('='*100)

total_tests = len(phase_summary_df)
total_phases = len(phase_results)
total_plots = len(list(PLOTS_DIR.glob('**/*.png')))
total_csvs = len(list(RESULTS_DIR.glob('**/*.csv')))

print(f'\nPrzeprowadzono {total_phases} faz testowania')
print(f'Laczna liczba testow: {total_tests}')
print(f'Liczba modeli: {len(loaded_models)}')
print(f'Warianty zbiorow danych: {len(test_datasets)}')
print(f'Wygenerowano wizualizacji: {total_plots}')
print(f'Wygenerowano plikow CSV: {total_csvs}')

print('='*100)
print('TOP 3 MODELE (OGOLNIE - WSZYSTKIE FAZY I ROZKLADY)')
print('='*100)

model_avg_scores = ranking_df.groupby('Model').agg({
    'Wynik rankingowy': 'mean',
    'ROC-AUC': 'mean',
    'F1': 'mean',
    'Dokładność': 'mean'
}).reset_index()

model_avg_scores = model_avg_scores.sort_values('Wynik rankingowy', ascending=False)

for i, (idx, row) in enumerate(model_avg_scores.head(3).iterrows()):
    medal = 'GOLD' if i == 0 else 'SILVER' if i == 1 else 'BRONZE'
    print(f"{medal} {row['Model']}")
    print(f"   Średni wynik rankingowy: {row['Wynik rankingowy']:.4f}")
    print(f"   Średnie ROC-AUC: {row['ROC-AUC']:.4f} | Średnie F1: {row['F1']:.4f} | Średnia dokładność: {row['Dokładność']:.4f}")
    print()

print('='*100)
print('TOP 3 MODELE DLA KAZDEJ FAZY (ROZKLAD 50/50)')
print('='*100)
for phase_name in phase_summary_df['Faza'].unique():
    phase_50_50 = ranking_df[
        (ranking_df['Faza'] == phase_name) &
        (ranking_df['Rozkład'] == '50% phishing, 50% bezpieczne')
    ].head(3)

    print(f'\n{phase_name}:')
    for idx, row in enumerate(phase_50_50.iterrows(), 1):
        row_data = row[1]
        print(f"  {idx}. {row_data['Model']} ({row_data['Zbiór']}) - "
              f"Wynik: {row_data['Wynik rankingowy']:.4f} | ROC-AUC: {row_data['ROC-AUC']:.4f} | "
              f"F1: {row_data['F1']:.4f}")

print('\n' + '='*100)
print('PODSUMOWANIE WEDLUG ROZMIARU ZBIORU')
print('='*100)
size_summary = phase_summary_df.groupby('Faza').agg({
    'Dokladnosc': ['mean', 'std', 'min', 'max'],
    'F1': ['mean', 'std', 'min', 'max'],
    'ROC-AUC': ['mean', 'std', 'min', 'max'],
    'Bledne': 'sum'
}).round(4)
print(size_summary)

print('\n' + '='*100)
print('PODSUMOWANIE WEDLUG ROZKLADU KLAS')
print('='*100)
dist_summary = phase_summary_df.groupby('Rozklad').agg({
    'Dokladnosc': ['mean', 'std'],
    'F1': ['mean', 'std'],
    'ROC-AUC': ['mean', 'std'],
    'Bledne': 'sum'
}).round(4)
print(dist_summary)

print('\n' + '='*100)
print('PODSUMOWANIE WEDLUG MODELU')
print('='*100)
model_summary = phase_summary_df.groupby('Model').agg({
    'Dokladnosc': ['mean', 'std'],
    'F1': ['mean', 'std'],
    'ROC-AUC': ['mean', 'std'],
    'Bledne': 'sum'
}).round(4)
model_summary = model_summary.sort_values(('ROC-AUC', 'mean'), ascending=False)
print(model_summary)

print('\n' + '='*100)
print('STRUKTURA KATALOGOW Z WYNIKAMI')
print('='*100)
print(f'\nFolder {RESULTS_DIR}/')
print(f'  ├── summary/')
print(f'  │   ├── all_phases_results.csv')
print(f'  │   ├── complete_ranking.csv')
print(f'  │   ├── model_summary.csv')
print(f'  │   └── all_phases_misclassifications.csv')
for i in range(1, 6):
    phase_csv_count = len(list((RESULTS_DIR / f'phase_{i}').glob('*.csv')))
    print(f'  ├── phase_{i}/ ({phase_csv_count} plikow CSV)')
    print(f'  │   ├── phase_{i}_detailed_results.csv')
    print(f'  │   ├── phase_{i}_ranking.csv')
    print(f'  │   └── phase_{i}_misclassifications.csv')

print(f'\nFolder {PLOTS_DIR}/')
print(f'  ├── summary/')
print(f'  │   ├── size_impact_all_phases.png')
print(f'  │   └── distribution_impact_phase5.png')
for i in range(1, 6):
    phase_plot_count = len(list((PLOTS_DIR / f'phase_{i}').glob('*.png')))
    print(f'  ├── phase_{i}/ ({phase_plot_count} wizualizacji)')
    print(f'  │   ├── confusion_matrices_*.png (3 pliki)')
    print(f'  │   ├── metrics_comparison_*.png (3 pliki)')
    print(f'  │   ├── misclassification_dist_*.png (3 pliki)')
    print(f'  │   └── phase_{i}_statistics.png')

print('\n' + '='*100)
print('PODSUMOWANIE WYGENEROWANYCH PLIKOW')
print('='*100)
print(f'  Pliki CSV: {total_csvs}')
print(f'     • Wyniki fazowe: 5 plikow')
print(f'     • Rankingi fazowe: 5 plikow')
print(f'     • Bledne klasyfikacje fazowe: 5 plikow')
print(f'     • Podsumowania: 4 pliki w folderze summary')
print(f'\n  Wizualizacje: {total_plots}')
print(f'     • Macierze pomylek: ~15 plikow (3 na faze)')
print(f'     • Porownania metryk: ~15 plikow (3 na faze)')
print(f'     • Rozklady bledow: ~15 plikow (3 na faze)')
print(f'     • Statystyki fazowe: 5 plikow (1 na faze)')
print(f'     • Podsumowania: 2 pliki w folderze summary')

print('\nTestowanie fazowe zakonczone pomyslnie!')
print('Wszystkie wyniki zorganizowane w podfolderach wedlug faz')
print('='*100)